# 01 — Synthetic Data Generation**Rift: Graph ML for Fraud Detection, Replay, and Audit**This notebook demonstrates Rift's synthetic transaction generator with 7 injected fraud patterns:velocity bursts, unusual merchant shifts, geographic jumps, new device usage, coordinated device reuse, account takeover sequences, and testing transactions.[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/01_data_generation.ipynb)

## Environment Setup

In [ ]:
# Clone repo and install dependencies (run once)import subprocess, sys, osREPO = "https://github.com/AngelP17/Rift.git"if not os.path.exists("/content/Rift"):    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)os.chdir("/content/Rift")subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars", "numpy", "structlog", "python-dotenv"], check=False)sys.path.insert(0, "src")# Detect GPUtry:    import torch    print(f"Device: {"cuda" if torch.cuda.is_available() else "cpu"}")except ImportError:    print("Device: cpu (torch not installed)")

## Generate TransactionsThe generator creates entities (users, merchants, devices, accounts), samples transactions with realistic distributions, then injects fraud patterns at the specified rate.

In [ ]:
from data.generator import generate_transactionsdf = generate_transactions(    n_txns=50_000,    n_users=2_000,    n_merchants=500,    fraud_rate=0.03,    seed=42,)print(f"Shape: {df.shape}")print(f"Fraud rate: {df["is_fraud"].mean():.4f}")print(f"Columns: {df.columns}")df.head(10)

## Explore Transaction Distributions

In [ ]:
import polars as plprint("=== Amount Statistics ===")print(df["amount"].describe())print("=== Channel Distribution ===")print(df.group_by("channel").len().sort("len", descending=True))print("=== Currency Distribution ===")print(df.group_by("currency").len().sort("len", descending=True))print("=== MCC Distribution (top 10) ===")print(df.group_by("mcc").len().sort("len", descending=True).head(10))

## Analyze Fraud Patterns

In [ ]:
fraud = df.filter(pl.col("is_fraud") == 1)legit = df.filter(pl.col("is_fraud") == 0)print(f"Fraud transactions: {len(fraud):,}")print(f"Legit transactions: {len(legit):,}")print("=== Fraud Amount Distribution ===")print(fraud["amount"].describe())print("=== Legit Amount Distribution ===")print(legit["amount"].describe())print("=== Fraud by Channel ===")print(fraud.group_by("channel").len().sort("len", descending=True))print("=== Fraud by MCC ===")print(fraud.group_by("mcc").len().sort("len", descending=True).head(10))

## Temporal Distribution

In [ ]:
timestamps = df.sort("timestamp")print(f"Date range: {timestamps["timestamp"].min()} to {timestamps["timestamp"].max()}")# Fraud over time (by week)weekly = (    df.with_columns(pl.col("timestamp").dt.truncate("1w").alias("week"))    .group_by("week")    .agg([        pl.col("is_fraud").sum().alias("fraud_count"),        pl.col("is_fraud").len().alias("total"),        pl.col("is_fraud").mean().alias("fraud_rate"),    ])    .sort("week"))print("=== Weekly Fraud Rate ===")print(weekly)

## Save Dataset

In [ ]:
from utils.io import save_parquetpath = save_parquet(df, "transactions")print(f"Saved to: {path}")print(f"File size: {path.stat().st_size / 1024:.1f} KB")

## Reproducibility Check

In [ ]:
df2 = generate_transactions(n_txns=50_000, n_users=2_000, n_merchants=500, fraud_rate=0.03, seed=42)assert df["amount"].to_list() == df2["amount"].to_list(), "Reproducibility check failed!"print("Reproducibility check passed: identical outputs with same seed")